In [ ]:
import os, pathlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, ConfusionMatrixDisplay
SEED = 42
tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
print('TensorFlow:', tf.__version__)

In [ ]:
(raw_train, raw_val, raw_test), ds_info = tfds.load(
    'cats_vs_dogs', split=['train[:20%]', 'train[70%:75%]', 'train[95%:]'],
    as_supervised=True, shuffle_files=True, with_info=True)
class_names = ds_info.features['label'].names
print('Clases:', class_names)
print('Ejemplo de especificación:', raw_train.element_spec)
for name, ds in [('train', raw_train), ('validation', raw_val), ('test', raw_test)]:
    print(name, tf.data.experimental.cardinality(ds).numpy())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for ax, (img, label) in zip(axes.ravel(), raw_train.take(8)):
    ax.imshow(img.numpy())
    ax.set_title(f'{class_names[int(label)]}\n{tuple(img.shape)}')
    ax.axis('off')
plt.tight_layout(); plt.show()


sample_labels = [int(y.numpy()) for _, y in raw_train.take(500)]
print(pd.Series(sample_labels).map(dict(enumerate(class_names))).value_counts())

In [ ]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.int32)

def augment(image, label):
    image = tf.image.random_brightness(image, max_delta=0.12)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def make_dataset(raw, training=False):
    ds = raw.map(preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(raw_train, training=True)
val_ds = make_dataset(raw_val)
test_ds = make_dataset(raw_test)
for x_batch, y_batch in train_ds.take(1):
    print('x:', x_batch.shape, x_batch.dtype, '| y:', y_batch.shape, y_batch.dtype)
    print('Rango de píxeles:', float(tf.reduce_min(x_batch)), float(tf.reduce_max(x_batch)))

In [ ]:
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(images[k].numpy())
    ax.set_title(class_names[int(labels[k])]); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
BACKBONE =  'VGG16'#'ResNet50'
RUN_IMAGENET_INFERENCE = True
RUN_SCRATCH = False  
RUN_FINETUNE = True
EPOCHS_HEAD = 1
EPOCHS_FT = 1
MAX_EPOCHS_TUNER = 2
OUT_DIR = pathlib.Path('resultados_resnet_vgg_catsdogs')
OUT_DIR.mkdir(exist_ok=True, parents=True)

APPLICATIONS = {
    'ResNet50': (tf.keras.applications.ResNet50,
                 tf.keras.applications.resnet50.preprocess_input,
                 tf.keras.applications.resnet50.decode_predictions),
    'VGG16': (tf.keras.applications.VGG16,
              tf.keras.applications.vgg16.preprocess_input,
              tf.keras.applications.vgg16.decode_predictions),
}
assert BACKBONE in APPLICATIONS
print('Modelo:', BACKBONE)

In [ ]:
import glob
def imagenet_inference(name=BACKBONE,folder=False):
    constructor, preprocess_fn, decoder = APPLICATIONS[name]
    imagenet_model = constructor(weights='imagenet', include_top=True)
    if folder:
        animls = list(glob.glob('*.jpg'))
        image_raw = tf.io.read_file(animls[0])
        images = tf.image.decode_jpeg(image_raw, channels=3) # channels=3 asegura RGB
        image = tf.image.resize(images, [224, 224])
        image = tf.expand_dims(image, axis=0)
    else:
        images, _ = next(iter(test_ds))
        image = tf.image.resize(images[:1]*255., (224, 224))
    print(image.shape)
    scores = imagenet_model.predict(preprocess_fn(tf.identity(image)), verbose=0)
    plt.figure(figsize=(4, 4))
    plt.imshow(tf.clip_by_value(image[0] / 255.0, 0, 1))
    plt.axis('off'); plt.title(f'{name} — ImageNet'); plt.show()
    print('Top 3:', decoder(scores, top=1)[0])
    del imagenet_model

if RUN_IMAGENET_INFERENCE:
    imagenet_inference(folder=False)

In [ ]:
def build_transfer_model(name=BACKBONE, weights='imagenet'):
    constructor, preprocess_fn, _ = APPLICATIONS[name]

    backbone = constructor(include_top=False,
                           weights=weights,
                           input_shape=(*IMAGE_SIZE, 3))
    
    backbone.trainable = (weights is None)

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='imagen')
    x = layers.Rescaling(255.0, name='restore_pixel_scale')(inputs)# ImageNet necesita [0,255] antes de preprocess_input
    x = layers.Lambda(preprocess_fn, name='preprocess_imagenet')(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    outputs = layers.Dense(2, activation='softmax', name='probabilidades')(x)
    model = tf.keras.Model(inputs, outputs, name=name + '_Gatos_Perros')
    return model, backbone

In [ ]:
scratch_model, scratch_backbone = build_transfer_model(weights=None)
scratch_model.summary()
scratch_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

In [ ]:
scratch_history = scratch_model.fit(train_ds, validation_data=val_ds, steps_per_epoch=40,verbose=1,validation_steps=40)

In [ ]:
results = scratch_model.evaluate(test_ds, return_dict=True)
print('Métricas test:', results)
y_true = np.concatenate([yb.numpy() for _, yb in test_ds])
pred_prob = scratch_model.predict(test_ds, verbose=0)
y_pred = np.argmax(pred_prob, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print('ROC-AUC:', roc_auc_score(y_true, pred_prob[:, 1]))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — test'); plt.show()

In [ ]:
# Comparar probabilidades en unas pocas imágenes del conjunto de prueba
sample_x, sample_y = next(iter(test_ds))
sample_prob = scratch_model.predict(sample_x, verbose=0)
fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in enumerate(axs.ravel()):
    ax.imshow(sample_x[k])
    idx = int(np.argmax(sample_prob[k]))
    ax.set_title(f'Real: {class_names[int(sample_y[k])]}\nPred: {class_names[idx]} ({sample_prob[k,idx]:.2f})')
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
TL_model, TL_backbone = build_transfer_model()
TL_model.summary()
TL_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

In [ ]:
TL_history = TL_model.fit(train_ds, validation_data=val_ds, steps_per_epoch=40*2,verbose=1,validation_steps=40)

In [ ]:
results = TL_model.evaluate(test_ds, return_dict=True)
print('Métricas test:', results)
y_true = np.concatenate([yb.numpy() for _, yb in test_ds])
pred_prob = TL_model.predict(test_ds, verbose=0)
y_pred = np.argmax(pred_prob, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print('ROC-AUC:', roc_auc_score(y_true, pred_prob[:, 1]))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — test'); plt.show()

In [ ]:
def enable_finetuning(backbone, last_n=15):
    backbone.trainable = True
    for layer in backbone.layers[:-last_n]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    print('Capas backbone entrenables:', sum(layer.trainable for layer in backbone.layers),
          '/', len(backbone.layers))

ft_model, ft_backbone = build_transfer_model(BACKBONE, weights='imagenet')
enable_finetuning(ft_backbone, last_n=2)

ft_model.summary()
ft_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

In [ ]:
ft_history = ft_model.fit(train_ds, validation_data=val_ds, steps_per_epoch=40,verbose=1,validation_steps=40)

In [ ]:
results = ft_model.evaluate(test_ds, return_dict=True)
print('Métricas test:', results)
y_true = np.concatenate([yb.numpy() for _, yb in test_ds])
pred_prob = ft_model.predict(test_ds, verbose=0)
y_pred = np.argmax(pred_prob, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print('ROC-AUC:', roc_auc_score(y_true, pred_prob[:, 1]))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — test'); plt.show()

In [ ]:
sample_x, sample_y = next(iter(test_ds))
sample_prob = ft_model.predict(sample_x, verbose=0)
fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in enumerate(axs.ravel()):
    ax.imshow(sample_x[k])
    idx = int(np.argmax(sample_prob[k]))
    ax.set_title(f'Real: {class_names[int(sample_y[k])]}\nPred: {class_names[idx]} ({sample_prob[k,idx]:.2f})')
    ax.axis('off')
plt.tight_layout(); plt.show()

# Multi-GPU y capacitación distribuida
Para hacer un entrenamiento síncrono de un solo host y multidispositivo con un modelo de Keras, lo usaría El tf.distribute.MirroredStrategyAPI. Así es como funciona:

    Instanciar a MirroredStrategy, configurando opcionalmente qué dispositivos específicos usted Quiere utilizar (por defecto la estrategia utilizará todas las GPU disponibles).

In [ ]:
def build_transfer_model(name=BACKBONE, weights='imagenet'):
    constructor, preprocess_fn, _ = APPLICATIONS[name]

    backbone = constructor(include_top=False,
                           weights=weights,
                           input_shape=(*IMAGE_SIZE, 3))
    
    backbone.trainable = (weights is None)

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='imagen')
    x = layers.Rescaling(255.0, name='restore_pixel_scale')(inputs)# ImageNet necesita [0,255] antes de preprocess_input
    x = layers.Lambda(preprocess_fn, name='preprocess_imagenet')(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    outputs = layers.Dense(2, activation='softmax', name='probabilidades')(x)
    model = tf.keras.Model(inputs, outputs, name=name + '_Gatos_Perros')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
    return model, backbone
strategy = tf.distribute.MirroredStrategy()
print("Number of devices: {}".format(strategy.num_replicas_in_sync))
with strategy.scope():
    model_multi,_ = build_transfer_model(BACKBONE)

In [ ]:
model_multi.fit(train_ds, validation_data=val_ds, epochs=1)

In [ ]:
model_multi.evaluate(test_ds)

## Fine-tuning con Hugging Face 🤗 (PyTorch)

Continuamos el mismo problema gatos vs perros, ahora con la librería `transformers` de Hugging Face:

1. Se carga el dataset **`microsoft/cats_vs_dogs`** directamente del Hub.
2. Se carga un modelo pre-entrenado del Hub (`AutoModelForImageClassification`) y se cambia su cabeza de 1000 clases ImageNet por una de 2 clases.
3. Igual que con `enable_finetuning`, se congela el backbone y se descongelan solo los últimos bloques.
4. Se entrena con `Trainer` y se evalúa en test con las mismas métricas.
#### !pip install -q torch torchvision "transformers>=4.46" "datasets>=2.20" accelerate

In [ ]:
import io, gc,types
import torch
import transformers
from datasets import Dataset, DatasetDict, Features, ClassLabel, Image as HFImage
from transformers import (AutoImageProcessor, AutoModelForImageClassification,
                          TrainingArguments, Trainer, set_seed, pipeline)
from PIL import Image as PILImage

set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| transformers:', transformers.__version__, '| dispositivo:', DEVICE)

# Liberar memoria de los modelos Keras (opcional; útil si se comparte la GPU con TensorFlow)
tf.keras.backend.clear_session(); gc.collect()

###### https://huggingface.co/microsoft/resnet-50

In [ ]:
HF_CHECKPOINTS = {
    'ResNet50': 'microsoft/resnet-50',   
}
HF_MODEL = 'ResNet50'
HF_CHECKPOINT = HF_CHECKPOINTS[HF_MODEL]
HF_TRAIN_LAST_N = 2     
HF_MAX_STEPS = 160     
HF_BATCH_SIZE = 32
HF_LR = 5e-5           
HF_LR_HEAD = 1e-3     
HF_TRAIN_SIZE = 4000  
HF_VAL_FRAC = 0.05     
HF_TEST_FRAC = 0.05    
HF_OUT = OUT_DIR / f'hf_{HF_MODEL}_catsdogs'
print('Checkpoint:', HF_CHECKPOINT)

In [ ]:
from datasets import load_dataset

hf_full = load_dataset('microsoft/cats_vs_dogs', split='train').rename_column('labels', 'label')
print(hf_full)
print('Clases:', hf_full.features['label'].names)


split = hf_full.train_test_split(test_size=0.1,seed=SEED)
val_test = split['test'].train_test_split(test_size=0.5, seed=SEED)
hf_train = split['train'].shuffle(seed=SEED)

hf_ds = DatasetDict(train=hf_train, validation=val_test['train'], test=val_test['test'])
print(hf_ds)
for name, d in hf_ds.items():
    print(name, dict(zip(d.features['label'].names, np.bincount(d['label']).tolist())))


In [ ]:
processor = AutoImageProcessor.from_pretrained(HF_CHECKPOINT)
print(processor)


def transform(batch):
    imgs = [im.convert('RGB') for im in batch['image']]
    return {'pixel_values': processor(images=imgs, return_tensors='pt')['pixel_values'],
            'labels': batch['label']}


hf_ds['train'].set_transform(transform)
hf_ds['validation'].set_transform(transform)
hf_ds['test'].set_transform(transform)

def collate_fn(examples):
    return {'pixel_values': torch.stack([e['pixel_values'] for e in examples]),
            'labels': torch.tensor([e['labels'] for e in examples])}

ex = hf_ds['train'][0]
print('pixel_values:', tuple(ex['pixel_values'].shape), '| label:', class_names[ex['labels']])

In [ ]:
id2label = dict(enumerate(class_names))
label2id = {v: k for k, v in id2label.items()}

hf_model = AutoModelForImageClassification.from_pretrained(
    HF_CHECKPOINT, id2label=id2label, label2id=label2id,  
    ignore_mismatched_sizes=True)   

def hf_enable_finetuning(model, last_n=2):
    for p in model.resnet.parameters():
        p.requires_grad = False
    blocks = [blk for stage in model.resnet.encoder.stages for blk in stage.layers]
    for blk in (blocks[-last_n:] if last_n > 0 else []):
        for p in blk.parameters():
            p.requires_grad = True
            
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'Parámetros entrenables: {n_train:,} / {n_total:,} ({100 * n_train / n_total:.1f} %)')

def hf_freeze_batchnorm(model):
    n = 0
    for m in model.modules():
        if isinstance(m, torch.nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.train = types.MethodType(lambda self, mode=True: torch.nn.Module.train(self, False), m)
            m.eval(); n += 1
    print('BatchNorm congeladas:', n)

hf_enable_finetuning(hf_model, last_n=HF_TRAIN_LAST_N)
hf_freeze_batchnorm(hf_model) 

In [ ]:
def hf_compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {'accuracy': float((np.argmax(logits, axis=1) == labels).mean())}

hf_args = TrainingArguments(
    output_dir=str(HF_OUT),
    per_device_train_batch_size=HF_BATCH_SIZE,
    per_device_eval_batch_size=HF_BATCH_SIZE,
    learning_rate=HF_LR,
    max_steps=HF_MAX_STEPS,
    eval_strategy='steps',
     remove_unused_columns=False,
    eval_steps=max(1, HF_MAX_STEPS // 2),
    seed=SEED,
)




hf_trainer = Trainer(model=hf_model, args=hf_args, data_collator=collate_fn,
                     train_dataset=hf_ds['train'], eval_dataset=hf_ds['validation'],
                     compute_metrics=hf_compute_metrics)
hf_train_result = hf_trainer.train()
print(hf_train_result.metrics)

In [ ]:
hf_pred = hf_trainer.predict(hf_ds['test'])
print('Métricas test:', hf_pred.metrics)
y_true = hf_pred.label_ids
pred_prob = torch.softmax(torch.tensor(hf_pred.predictions), dim=1).numpy()
y_pred = np.argmax(pred_prob, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print('ROC-AUC:', roc_auc_score(y_true, pred_prob[:, 1]))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', values_format='d')
plt.title(f'Matriz de confusión — test ({HF_MODEL} Hugging Face)'); plt.show()

In [ ]:
raw_test_imgs = hf_ds['test'].with_format(None)   # sin transform, para ver la imagen original
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(raw_test_imgs), size=8, replace=False)
fig, axs = plt.subplots(2, 4, figsize=(13, 7))
for k, ax in zip(sample_idx, axs.ravel()):
    k = int(k)
    real, pred = int(y_true[k]), int(y_pred[k])
    ax.imshow(raw_test_imgs[k]['image'])
    ax.set_title(f'Real: {class_names[real]}\nPred: {class_names[pred]} ({pred_prob[k, pred]:.2f})',
                 color='green' if real == pred else 'red')
    ax.axis('off')
fig.suptitle(f'{HF_MODEL} — accuracy test: {(y_pred == y_true).mean():.3f}  |  '
             f'predicciones por clase: {dict(zip(class_names, np.bincount(y_pred, minlength=2).tolist()))}')
plt.tight_layout(); plt.show()

In [ ]:
hf_trainer.save_model(str(HF_OUT))
processor.save_pretrained(str(HF_OUT))
print('Modelo guardado en', HF_OUT)

clf = pipeline('image-classification', model=str(HF_OUT), device=DEVICE)
animal_files = sorted(glob.glob('/home/csmhhort/figura1_perro.jpg'))
if animal_files:
    img = PILImage.open(animal_files[0]).convert('RGB')
    plt.figure(figsize=(4, 4)); plt.imshow(img); plt.axis('off'); plt.title(pathlib.Path(animal_files[0]).name); plt.show()
    print(clf(img))
else:
    print(clf(raw_test_imgs[0]['image']))

In [ ]:
# from huggingface_hub import login
# login()   # te pide el token

# REPO_ID = 'nombre_usuario/resnet50-cats-dogs' 

# hf_model.push_to_hub(REPO_ID)
# processor.push_to_hub(REPO_ID)

#### Se puede  usar desde cualquier sitio:
# clf = pipeline('image-classification', model='tu_usuario/resnet50-cats-dogs')